In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv("../data/processed/bank_churn_features.csv")

X = df.drop(columns=["Exited"])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(y_test.value_counts())

(8000, 15) (2000, 15)
Exited
0    1593
1     407
Name: count, dtype: int64


In [2]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                max_depth=3, random_state=42)
gb.fit(X_train, y_train)

probs_gb = gb.predict_proba(X_test)[:, 1]
pred_gb = (probs_gb >= 0.5).astype(int)

print(classification_report(y_test, pred_gb, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, probs_gb), 3))

              precision    recall  f1-score   support

           0      0.880     0.963     0.919      1593
           1      0.770     0.484     0.594       407

    accuracy                          0.866      2000
   macro avg      0.825     0.723     0.757      2000
weighted avg      0.857     0.866     0.853      2000

ROC-AUC: 0.867


In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

forest = RandomForestClassifier(n_estimators=200, max_depth=8,
                                random_state=42, n_jobs=-1)

rf_scores = cross_val_score(forest, X_train, y_train, cv=cv, scoring="roc_auc")
gb_scores = cross_val_score(gb, X_train, y_train, cv=cv, scoring="roc_auc")

print("Random Forest:", rf_scores.round(3), "mean:", rf_scores.mean().round(3))
print("Gradient Boosting:", gb_scores.round(3), "mean:", gb_scores.mean().round(3))

Random Forest: [0.858 0.864 0.872 0.839 0.855] mean: 0.858
Gradient Boosting: [0.862 0.873 0.871 0.849 0.856] mean: 0.862


In [4]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(gb, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    flag = (oof >= t)
    caught = (flag & (y_train == 1)).sum()
    wasted = (flag & (y_train == 0)).sum()
    missed = ((~flag) & (y_train == 1)).sum()
    rows.append({"cutoff": t,
                 "flagged": caught + wasted,
                 "recall": round(caught / (caught + missed), 3),
                 "precision": round(caught / (caught + wasted), 3),
                 "wasted_calls": wasted,
                 "missed_churners": missed})

pd.DataFrame(rows)

,cutoff,flagged,recall,precision,wasted_calls,missed_churners
0,0.50,1027,0.481,0.763,243,846
1,0.40,1294,0.547,0.689,403,739
2,0.30,1688,0.635,0.613,653,595
3,0.25,1983,0.685,0.563,867,514
4,0.20,2392,0.745,0.508,1177,415


In [5]:
from sklearn.metrics import confusion_matrix

pred_03 = (probs_gb >= 0.3).astype(int)

print(classification_report(y_test, pred_03, digits=3))
print(confusion_matrix(y_test, pred_03))

              precision    recall  f1-score   support

           0      0.910     0.899     0.905      1593
           1      0.623     0.654     0.638       407

    accuracy                          0.849      2000
   macro avg      0.767     0.776     0.771      2000
weighted avg      0.852     0.849     0.850      2000

[[1432  161]
 [ 141  266]]


In [6]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    {"model": gb, "cutoff": 0.30, "features": list(X_train.columns)},
    "../models/churn_gb_model.joblib",
)

saved = joblib.load("../models/churn_gb_model.joblib")
print(saved["cutoff"], len(saved["features"]))

0.3 15
